In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
from scipy.ndimage import median_filter
from scipy.signal import welch, csd
from scipy.optimize import curve_fit
import glob
from windrose import WindroseAxes
import cmcrameri as cmc
from cmcrameri import cm
from scipy.stats import linregress
import sys
import os
from pathlib import Path
import pickle
from datetime import datetime

# Add project 'src' directory to sys.path when running from the notebooks/ folder
# (notebooks/ is expected to be inside the repo; repo_root = parent of cwd)
repo_root = Path.cwd().parent
src_path = str(repo_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import modules from the installed package. If this fails, the editable install
# (`pip install -e .`) may be missing or kernel needs restart.
try:
    from ec.func_read_data import *
    from ec.func_coherence import *
    from mo.func_mo import *
    from spc.normalize import *
    from plotting.funcs_plots import *

    from ec.sensor_info import *
except Exception as e:
    print('Package import failed:', e)
    print('Make sure you ran `pip install -e .` (editable install) and restart the kernel, or that src/ exists at:', src_path)
else:
    # For development: auto-reload modules (keep this enabled while editing source files)
    %load_ext autoreload
    %autoreload 2

In [2]:
"""READ AND CLEAN SLOWDATA"""
folder='/home/engbers/Documents/PhD/EC_data_convert/converted/'
sensor='SFC'
start='2024-01-01 00:00'
end='2025-01-01 00:00'

"""GET SENSOR INFO"""
plim, calibration_coefficients, heights = get_sensor_info(sensor, 2024)

"""READ FAST AND SLOW DATA from folder"""
slowdata=read_data(folder, 'slow', sensor, start, end, plot_data=False)

"""CLEAN SLOWDATA"""
slowdata_cleaned=clean_slowdata(slowdata)

Using 2024 calibration coefficients
{'A': 4820.04, 'B': 3792900.0, 'C': -115477000.0, 'H2O_Zero': 0.7087, 'H20_Span': 0.9885}
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20240118_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20240120_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20240122_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20240201_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20240205_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20240214_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20240220_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20241223_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20250208_SFC
Reading data from /home/engbers/Documents/PhD/EC_data_convert/converted/20250211_SFC
Reading data from /home/

In [ ]:
"""PROCESS SPC SCRIPTS AND SAVE TO ONE MINUTE FILES"""
SPC_folder = '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed'
SPC_filenames = []
SPC_all = pd.DataFrame()
file_list = sorted(glob.glob(os.path.join(SPC_folder, '*.txt')))
batch_size = 25

for i in range(0, len(file_list), batch_size):
    batch_files = file_list[i:i + batch_size]
    # SPC_filenames.append(file)
    print(batch_files)
    SPC = getNormalizedData(batch_files, slowdata_cleaned)
    SPC = SPC.resample('1min').mean()
    SPC.to_csv('/home/engbers/Documents/PhD/EC_data_convert/SPC/SPC_OneMin_processed/'+batch_files[0].split('/')[-1], index=True, sep='\t')

['/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240220_082027Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240221_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240222_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240223_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240224_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240224_084612Snow-04-1sec-org-0001.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240225_000001Snow-04-1sec-org-0001.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240226_000001Snow-04-1sec-org-0001.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240227_000001S

/home/engbers/Documents/Github/DataProcessingScripts/src/spc/normalize.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corrected_data = grouped_raw_data.apply(correctTempBloc,conversions_df) #Apply correction by bloc of temperatures


['/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240313_000001Snow-04-1sec-org-0003.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240314_000001Snow-04-1sec-org-0003.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240315_000001Snow-04-1sec-org-0003.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240316_000001Snow-04-1sec-org-0003.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240316_090106Snow-04-1sec-org-0004.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240317_000001Snow-04-1sec-org-0004.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240318_000001Snow-04-1sec-org-0004.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240319_000001Snow-04-1sec-org-0004.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240320_000001S

/home/engbers/Documents/Github/DataProcessingScripts/src/spc/normalize.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corrected_data = grouped_raw_data.apply(correctTempBloc,conversions_df) #Apply correction by bloc of temperatures


['/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240404_000001Snow-04-1sec-org-0006.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240405_000001Snow-04-1sec-org-0006.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240406_000001Snow-04-1sec-org-0006.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240406_091601Snow-04-1sec-org-0007.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240407_000001Snow-04-1sec-org-0007.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240408_000001Snow-04-1sec-org-0007.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240409_000001Snow-04-1sec-org-0007.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240410_000001Snow-04-1sec-org-0007.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240411_000001S

...

/home/engbers/Documents/Github/DataProcessingScripts/src/spc/normalize.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corrected_data = grouped_raw_data.apply(correctTempBloc,conversions_df) #Apply correction by bloc of temperatures


['/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241129_000001Snow-04-1sec-org-0029.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241130_000001Snow-04-1sec-org-0029.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241201_000001Snow-04-1sec-org-0029.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241201_114000Snow-04-1sec-org-002A.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241202_000001Snow-04-1sec-org-002A.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241203_000001Snow-04-1sec-org-002A.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241204_000001Snow-04-1sec-org-002A.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241205_000001Snow-04-1sec-org-002A.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241205_151406S

/home/engbers/Documents/Github/DataProcessingScripts/src/spc/normalize.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corrected_data = grouped_raw_data.apply(correctTempBloc,conversions_df) #Apply correction by bloc of temperatures


['/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241220_000001Snow-04-1sec-org-002D.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241221_000001Snow-04-1sec-org-002D.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241222_000001Snow-04-1sec-org-002D.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U241223_000001Snow-04-1sec-org-002D.txt']


/home/engbers/Documents/Github/DataProcessingScripts/src/spc/normalize.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corrected_data = grouped_raw_data.apply(correctTempBloc,conversions_df) #Apply correction by bloc of temperatures


In [ ]:
"""PROCESS SPC SCRIPTS AND SAVE TO ONE MINUTE FILES"""
SPC_folder = '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed'
SPC_filenames = []
SPC_all = pd.DataFrame()
file_list = sorted(glob.glob(os.path.join(SPC_folder, '*.txt')))
batch_size = 25

for i in range(0, len(file_list), batch_size):
    batch_files = file_list[i:i + batch_size]
    # SPC_filenames.append(file)
    print(batch_files)
    SPC_filenames.extend(batch_files)
SPC_raw=getRawData(SPC_filenames)

['/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240220_082027Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240221_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240222_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240223_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240224_000001Snow-04-1sec-org-0000.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240224_084612Snow-04-1sec-org-0001.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240225_000001Snow-04-1sec-org-0001.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240226_000001Snow-04-1sec-org-0001.txt', '/home/engbers/Documents/PhD/EC_data_convert/SPC/20241223_SFC/processed/U240227_000001S

In [ ]:
"""READ SPC FILES"""
SPC_filenames = []
SPC_folder='/home/engbers/Documents/PhD/EC_data_convert/SPC/SPC_OneMin_processed/'
file_list = sorted(glob.glob(os.path.join(SPC_folder, '*.txt')))
for file in file_list:
    SPC_filenames.append(file)
SPC=getRawData(SPC_filenames)

In [ ]:
column_sums_SPC = []
column_sums_SPC_raw = []

# Calculate the sum of all values for each column from index 1 to -2 for SPC
for column in SPC.columns[1:-1]:
    column_sums_SPC.append(SPC[column].sum())

# Calculate the sum of all values for each column from index 1 to -2 for SPC_raw
for column in SPC_raw.columns[2:-1]:
    column_sums_SPC_raw.append(SPC_raw[column].sum())

# Create the combined histogram
plt.figure(figsize=(12, 6))
plt.bar(range(1, len(column_sums_SPC) + 1), column_sums_SPC, color='blue', edgecolor='black', alpha=0.7, label='SPC')
plt.bar(range(1, len(column_sums_SPC_raw) + 1), column_sums_SPC_raw, color='orange', edgecolor='black', alpha=0.5, label='SPC_raw')
plt.title("Combined Histogram of SPC and SPC_raw Columns")
plt.xlabel("Particle size [micrometers]")
plt.ylabel("Sum")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(range(1, len(column_sums_SPC) + 1), SPC.columns[1:-1], rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'SPC_raw' is not defined